In [0]:
from pyspark.sql.functions import max as spark_max, col, lit
from pyspark.sql.types import TimestampType

silver_table = "observatorio_dev.silver.agentes"

last_ingestion_timestamp = (
    spark.table(silver_table)
    .select(spark_max("ingestion_timestamp").alias("last_ingestion_timestamp"))
    .collect()[0]["last_ingestion_timestamp"]
)

print("Último ingestion_timestamp cargado en Silver:", last_ingestion_timestamp)

In [0]:
bronze_table = "observatorio_dev.bronze.agentes"

bronze_df = spark.table(bronze_table)

if last_ingestion_timestamp is not None:
    bronze_df = bronze_df.filter(
        col("ingestion_timestamp") > lit(last_ingestion_timestamp)
    )

display(bronze_df)

In [0]:
from pyspark.sql.functions import col, trim, upper, to_date, current_timestamp

silver_df = (
    bronze_df
    .select(
        to_date(col("fecha")).alias("fecha"),
        trim(upper(col("codigo_duracion"))).alias("codigo_duracion"),
        trim(upper(col("codigo_sic_agente"))).alias("codigo_agente"),
        trim(upper(col("nombre_agente"))).alias("nombre_agente"),
        trim(upper(col("actividad_agente"))).alias("actividad_agente"),

        col("source_file_name"),
        col("source_file_path"),
        col("ingestion_timestamp"),
        col("load_date"),

        current_timestamp().alias("silver_created_at"),
        current_timestamp().alias("silver_updated_at")
    )
    .dropDuplicates([
        "fecha",
        "codigo_duracion",
        "codigo_agente",
        "actividad_agente"
    ])
)

In [0]:
display(silver_df)

In [0]:
print("Registros Bronze:", bronze_df.count())
print("Registros Silver transformados:", silver_df.count())

In [0]:
from pyspark.sql.functions import count, when

silver_df.select(
    count(when(col("fecha").isNull(), True)).alias("fecha_nulls"),
    count(when(col("codigo_agente").isNull(), True)).alias("codigo_agente_nulls"),
    count(when(col("nombre_agente").isNull(), True)).alias("nombre_agente_nulls"),
    count(when(col("actividad_agente").isNull(), True)).alias("actividad_agente_nulls")
).show()

In [0]:
new_records_count = silver_df.count()

print("Registros nuevos a procesar:", new_records_count)

if new_records_count == 0:
    print("No hay registros nuevos para cargar en Silver.")
else:
    from delta.tables import DeltaTable

    target_table = "observatorio_dev.silver.agentes"
    target = DeltaTable.forName(spark, target_table)

    (
        target.alias("target")
        .merge(
            silver_df.alias("source"),
            """
            target.fecha = source.fecha
            AND target.codigo_duracion = source.codigo_duracion
            AND target.codigo_agente = source.codigo_agente
            AND target.actividad_agente = source.actividad_agente
            """
        )
        .whenMatchedUpdate(set={
            "nombre_agente": "source.nombre_agente",
            "source_file_name": "source.source_file_name",
            "source_file_path": "source.source_file_path",
            "ingestion_timestamp": "source.ingestion_timestamp",
            "load_date": "source.load_date",
            "silver_updated_at": "source.silver_updated_at"
        })
        .whenNotMatchedInsert(values={
            "fecha": "source.fecha",
            "codigo_duracion": "source.codigo_duracion",
            "codigo_agente": "source.codigo_agente",
            "nombre_agente": "source.nombre_agente",
            "actividad_agente": "source.actividad_agente",
            "source_file_name": "source.source_file_name",
            "source_file_path": "source.source_file_path",
            "ingestion_timestamp": "source.ingestion_timestamp",
            "load_date": "source.load_date",
            "silver_created_at": "source.silver_created_at",
            "silver_updated_at": "source.silver_updated_at"
        })
        .execute()
    )

    print("MERGE ejecutado correctamente.")

In [0]:
%sql
SELECT * FROM observatorio_dev.silver.agentes
LIMIT 10